# Forward Euler solver

For each step, $y_{n+1}=y_n+\Delta t f(t_n,y_n)$. All four variables use the same old state. The final step is shortened to reach the requested end time. Returned states have shape `(number_of_times, number_of_variables)`.

Stopping conditions are checked at step endpoints. Crossing time has a time-step-sized uncertainty; use step refinement before reporting boundaries. Local one-step error is $O(\Delta t^2)$ and global error is $O(\Delta t)$ for a smooth problem in the convergence regime.

In [2]:
import numpy as np

def forward_euler(f, initial_state, t_span, dt,
                  skip_altitude=150000.0):
    """Integrate the entry trajectory using Forward Euler."""

    if dt <= 0 or not np.isfinite(dt):
        raise ValueError("dt must be positive and finite.")

    t, t_end = t_span
    state = np.array(initial_state, dtype=float)

    times = [t]
    states = [state.copy()]

    entry_altitude = state[0]
    has_descended = False
    outcome = "unresolved"

    while True:
        h, v, gamma, s = state

        if not np.all(np.isfinite(state)) or v <= 0:
            outcome = "numerical failure"
            break

        if h <= 0:
            outcome = "surface"
            break

        if h < entry_altitude:
            has_descended = True

        # A skip requires descent followed by an upward crossing.
        if (has_descended and h >= skip_altitude
                and v * np.sin(gamma) > 0):
            outcome = "skip"
            break

        if t >= t_end:
            break

        # Shorten the last step to reach t_end.
        step = min(dt, t_end - t)

        # Evaluate all four derivatives using the current state.
        derivatives = np.asarray(f(t, state), dtype=float)

        # Update all four variables simultaneously.
        state = state + step * derivatives
        t = min(t + step, t_end)

        times.append(t)
        states.append(state.copy())

    return np.array(states), np.array(times), outcome